The imputation engine and settings follow the manuscript and the original `explore-predictors-ML3-imputation` chunk. Fitting uses the complete development partition, excludes future outcomes and retains forests and development donors for frozen application to holdout.
**Investigator decision, 2026-10-03.** Use the five completed development datasets for ML cross-validation; do not refit imputers within CV folds. Notebook 00 already used this fitting scope. The earlier fold-specific readmission tuning remains a superseded strategy. Notebooks 01 and 02 have since completed and received acceptance using the complete-development imputations; notebook 03 has completed TreeSHAP and its screening checkpoints. Saved historical results are retained, not relabelled as results of this workflow.
| Stage | Approved scope | Current status |
|---|---|---|
| Participant partition | Preserve the historical development/holdout split and cohort | Unchanged |
| Imputation | Five full-development missRanger fits, without outcomes | Saved fits reused only after provenance and training-data checks |
| Holdout and additional cohorts | Apply development forests and donors without refitting | Holdout completed; additional cohorts pending |
| ML cross-validation | Train and evaluate survival models in their original folds using the same completed development datasets | Tuning in 01 and 02 completed and accepted; 03 screening completed |
**Retained limitation of internal CV.** The five missRanger imputations were fitted once on the complete development sample, without outcome or follow-up variables, before survival-model cross-validation. Predictors from each internal validation fold therefore contributed to learning the imputer, although that fold is excluded from fitting its survival model. This CV does not independently validate the complete preprocessing-and-model-selection workflow. missRanger OOB diagnostics assess the imputation forests and do not remove this dependence. The direction and magnitude of its effect on the retained workflow have not been isolated by the available comparisons. The investigator explicitly retained this scope on 2026-10-05.
**Separate holdout boundary.** Holdout rows and outcomes were excluded from fitting the development imputers, ML tuning and SHAP screening. However, notebook 00's descriptive early-death audit inspected holdout outcomes, and inherited discovery decisions remain documented. Do not describe the entire historical and reconstructed workflow as wholly independent of the holdout.

Sources: [official missRanger reference](https://mayer79.github.io/missRanger/reference/missRanger.html) and [preprocessing and validation guidance](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).
Execution modes are explicit in `config.yml`. A full cached replay validates and reads existing fits and completed datasets; it does not count as a new imputation fit. Missing or incompatible caches must be reviewed before launching unplanned computation.


## 1. Configuration and official method


In [1]:
#| label: prep-config-and-environment
#| message: false
.t0 <- proc.time()
# Locate the only configuration file from either the project or notebook directory.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "config.yml"))) {
  parent <- dirname(root)
  if (identical(parent, root)) stop("config.yml was not found.", call. = FALSE)
  root <- parent
}
config <- yaml::read_yaml(file.path(root, "config.yml"))
resolve_path <- function(value) {
  absolute <- grepl("^([A-Za-z]:[/\\\\]|/|\\\\\\\\)", value)
  normalizePath(if (absolute) value else file.path(root, value), winslash = "/", mustWork = FALSE)
}
source_path <- function(name) resolve_path(config$sources[[name]])
object_dir <- resolve_path(config$paths$objects)
stopifnot(dir.exists(object_dir), config$preparation$mode %in% c("preflight", "pilot", "first_imputation", "full"))
# Limit threads before scientific namespaces load and verify the selected R environment.
threads <- as.integer(config$imputation$threads)
Sys.setenv(OMP_NUM_THREADS = threads, OPENBLAS_NUM_THREADS = threads, MKL_NUM_THREADS = threads, R_RANGER_NUM_THREADS = threads)
options(Ncpus = threads, ranger.num.threads = threads)
lock <- jsonlite::read_json(resolve_path(config$runtime$r_lock), simplifyVector = FALSE)
packages <- c("yaml", "jsonlite", "missRanger", "ranger", "nanoparquet", "digest", "ps")
stopifnot(as.character(getRversion()) == lock$R$Version)
display_packages <- c("htmltools", "knitr", "IRdisplay")
for (package in c(packages, display_packages)) {
  if (!requireNamespace(package, quietly = TRUE)) stop("Missing package: ", package, call. = FALSE)
  expected <- lock$Packages[[package]]$Version
  if (is.null(expected) || utils::packageVersion(package) != base::package_version(expected)) stop("Package version differs from lock: ", package, call. = FALSE)
}
if (requireNamespace("future", quietly = TRUE)) future::plan(future::sequential)
set.seed(as.integer(config$imputation$seed))
written <- character()
run_started <- Sys.time()
# Reuse the original htmltools/knitr table pattern, with scrolling and immediate notebook display.
show_table <- function(data, caption) {
  widget <- htmltools::browsable(htmltools::tags$div(class = "article2-table", role = "region", tabindex = 0, `aria-label` = caption, style = "max-height:420px;overflow:auto;max-width:100%;border:1px solid #ddd;padding:0.4rem;margin-bottom:0.8rem;", htmltools::HTML(knitr::kable(data, format = "html", caption = caption, row.names = FALSE, digits = 4, escape = TRUE))))
  IRdisplay::display_html(as.character(widget))
  invisible(widget)
}
show_table(data.frame(Package = c(packages, display_packages), Version = vapply(c(packages, display_packages), function(x) as.character(utils::packageVersion(x)), character(1L))), "Verified execution packages")
memory <- ps::ps_system_memory()
show_table(data.frame(Item = c("Total RAM (GiB)", "Available RAM (GiB)", "Threads"), Value = c(memory$total / 1024^3, memory$avail / 1024^3, threads)), "Resources before execution")
cat("Kernel process:", Sys.getpid(), "\n")
cat("Mode:", config$preparation$mode, "| Threads:", threads, "\n")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Package,Version
yaml,2.3.10
jsonlite,1.9.1
missRanger,2.6.1
ranger,0.17.0
nanoparquet,0.4.2
digest,0.6.37
ps,1.9.3
htmltools,0.5.8.1
knitr,1.50
IRdisplay,1.1


Item,Value
Total RAM (GiB),95.6941
Available RAM (GiB),54.5777
Threads,2.0000


Kernel process: 24880 


Mode: full | Threads: 2 


Elapsed: 0.019 minutes


In [2]:
#| label: prep-official-imputation-method
#| message: false
.t0 <- proc.time()
# Display the manuscript method and the authorized leakage correction before any cohort fit.
method <- data.frame(Component = c("Engine", "Imputations", "PMM donors", "Trees", "Maximum fitting iterations", "Unordered factors", "OOB diagnostic", "Historical OOB threshold", "Imputation seeds", "Development fitting", "Holdout application"), Current = c("missRanger", config$imputation$n_imputations, config$imputation$pmm_k, config$imputation$num_trees, config$imputation$maxiter, "order", "Selected-iteration OOB; summarize across imputations", config$imputation_diagnostics$oob_threshold, paste(seq.int(config$imputation$seed, length.out = config$imputation$n_imputations), collapse = ", "), "Complete development, no fold-wise refitting; six future outcome/time columns excluded", "Frozen development forests and donors"), row.names = NULL)
show_table(method, "Official imputation method and leakage correction")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Component,Current
Engine,missRanger
Imputations,5
PMM donors,15
Trees,200
Maximum fitting iterations,10
Unordered factors,order
OOB diagnostic,Selected-iteration OOB; summarize across imputations
Historical OOB threshold,0.7
Imputation seeds,"2125, 2126, 2127, 2128, 2129"
Development fitting,"Complete development, no fold-wise refitting; six future outcome/time columns excluded"


Elapsed: 0.000 minutes


## 2. Imputer controls


In [3]:
#| label: prep-imputer-guards
#| message: false
.t0 <- proc.time()
# Define future outcomes and metadata that must never enter an imputer.
outcome_names <- c("readmit_time_from_disch_m", "readmit_event", "death_time_from_disch_m", "death_event", "readmit_time_from_adm_m", "death_time_from_adm_m")
forbidden <- c(outcome_names, "center_id", "original_row_id", "legacy_row_id", "row_id", "hash_key", "is_train")
assert_predictors <- function(x) {
  bad <- intersect(names(x), forbidden)
  if (length(bad)) stop("Forbidden imputer columns: ", paste(bad, collapse = ", "), call. = FALSE)
  if (anyDuplicated(names(x)) || !ncol(x)) stop("Predictor columns must be unique and nonempty.", call. = FALSE)
  invisible(TRUE)
}
assert_observed <- function(original, completed) {
  stopifnot(identical(names(original), names(completed)), nrow(original) == nrow(completed), !anyNA(completed))
  valid <- vapply(names(original), function(v) {
    keep <- !is.na(original[[v]])
    identical(as.character(original[[v]][keep]), as.character(completed[[v]][keep]))
  }, logical(1L))
  if (!all(valid)) stop("Observed values changed: ", paste(names(valid)[!valid], collapse = ", "), call. = FALSE)
  invisible(TRUE)
}
# Keep the August method and its full prediction state, including development donors.
fit_imputer <- function(x, settings, seed) {
  assert_predictors(x)
  if (any(vapply(x, function(v) all(is.na(v)), logical(1L)))) stop("A development predictor is entirely missing.", call. = FALSE)
  missRanger::missRanger(data = x, formula = stats::as.formula(". ~ ."), pmm.k = settings$pmm_k, num.trees = settings$num_trees, maxiter = settings$maxiter, seed = seed, num.threads = settings$threads, keep_forests = TRUE, data_only = FALSE, returnOOB = TRUE, verbose = 1L, respect.unordered.factors = "order")
}
apply_imputer <- function(fit, x, settings, seed) {
  assert_predictors(x)
  stopifnot(identical(names(x), names(fit$data_raw)))
  unsupported <- setdiff(names(x)[colSums(is.na(x)) > 0L], fit$to_impute)
  if (length(unsupported)) stop("No fitted development imputation model for: ", paste(unsupported, collapse = ", "), call. = FALSE)
  for (v in names(x)) {
    if (is.factor(fit$data_raw[[v]])) {
      novel <- setdiff(unique(as.character(x[[v]][!is.na(x[[v]])])), levels(fit$data_raw[[v]]))
      if (length(novel)) stop("Unseen category in predictor: ", v, call. = FALSE)
      x[[v]] <- factor(as.character(x[[v]]), levels = levels(fit$data_raw[[v]]), ordered = is.ordered(fit$data_raw[[v]]))
    }
  }
  completed <- stats::predict(fit, newdata = x, pmm.k = settings$pmm_k, iter = settings$predict_iter, num.threads = settings$threads, seed = seed, verbose = 0L)
  assert_observed(x, completed)
  completed
}
# Extract the same final OOB vector returned by returnOOB=TRUE in the original workflow.
oob_report <- function(fit, imputation) {
  errors <- as.matrix(fit$pred_errors)
  selected <- as.integer(fit$best_iter)
  stopifnot(length(selected) == 1L, selected >= 1L, selected <= nrow(errors), !is.null(colnames(errors)))
  variables <- colnames(errors)
  regression <- vapply(fit$data_raw[variables], is.numeric, logical(1L))
  final <- data.frame(Imputation = imputation, Variable = variables, Scale = ifelse(regression, "Normalized MSE (1 - R^2)", "Misclassification proportion"), OOB = as.numeric(errors[selected, ]), Selected_iteration = selected, Missing_before = vapply(fit$data_raw[variables], function(x) sum(is.na(x)), integer(1L)), Above_threshold = as.numeric(errors[selected, ]) > config$imputation_diagnostics$oob_threshold, row.names = NULL)
  history <- data.frame(Imputation = imputation, Iteration = seq_along(fit$mean_pred_errors), Mean_OOB = as.numeric(fit$mean_pred_errors), Selected = seq_along(fit$mean_pred_errors) == selected)
  list(final = final, history = history)
}
missing_report <- function(before, after, imputation, partition) {
  data.frame(Imputation = imputation, Partition = partition, Variable = names(before), Missing_before = colSums(is.na(before)), Percent_before = 100 * colMeans(is.na(before)), Missing_after = colSums(is.na(after)), Percent_after = 100 * colMeans(is.na(after)), row.names = NULL)
}
# Refuse stale caches rather than silently refitting or overwriting them.
check_cache <- function(saved, current) {
  if (!identical(saved, current)) stop("Cache metadata mismatch.", call. = FALSE)
  invisible(TRUE)
}
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Elapsed: 0.000 minutes


In [4]:
#| label: prep-synthetic-frozen-imputer-control
#| message: false
.t0 <- proc.time()
# Exercise fit, frozen application and failure paths without using study participants.
i <- seq_len(120L)
toy <- data.frame(a = sin(i / 7) + i / 50, b = factor(rep(c("A", "B", "C"), length.out = 120L)), complete = cos(i / 11))
toy_dev <- toy[seq_len(96L), , drop = FALSE]
toy_new <- toy[97:120, , drop = FALSE]
toy_dev$a[seq(3L, 90L, 9L)] <- NA_real_
toy_dev$b[seq(4L, 92L, 8L)] <- NA
toy_new$a[c(2L, 5L, 9L)] <- NA_real_
toy_new$b[c(3L, 5L, 10L)] <- NA
small <- list(pmm_k = 3L, num_trees = 40L, maxiter = 3L, predict_iter = 3L, threads = threads)
toy_fit <- fit_imputer(toy_dev, small, 2125L)
toy_prediction <- apply_imputer(toy_fit, toy_new, small, 3131L)
assert_observed(toy_dev, toy_fit$data)
stopifnot(identical(toy_fit$data, fit_imputer(toy_dev, small, 2125L)$data))
stopifnot(identical(toy_prediction, apply_imputer(unserialize(serialize(toy_fit, NULL)), toy_new, small, 3131L)))
# Verify that outcome leakage, unsupported missingness and stale caches fail.
rejects <- function(expr) inherits(tryCatch(force(expr), error = identity), "error")
bad <- toy_dev
bad$death_event <- 0L
stopifnot(rejects(fit_imputer(bad, small, 2125L)))
bad <- toy_new
bad$complete[1L] <- NA_real_
stopifnot(rejects(apply_imputer(toy_fit, bad, small, 3131L)))
stopifnot(rejects(check_cache(list(seed = 1L), list(seed = 2L))))
# Verify the two disk formats with artificial data; remove only these temporary files.
local({
  model_file <- tempfile(fileext = ".rds")
  data_file <- tempfile(fileext = ".parquet")
  on.exit(unlink(c(model_file, data_file)), add = TRUE)
  saveRDS(toy_fit, model_file, compress = FALSE, version = 3L)
  stopifnot(identical(toy_prediction, apply_imputer(readRDS(model_file), toy_new, small, 3131L)))
  exchange <- toy_prediction
  exchange[] <- lapply(exchange, function(x) if (is.factor(x)) as.character(x) else x)
  rownames(exchange) <- NULL
  nanoparquet::write_parquet(exchange, data_file)
  stopifnot(identical(as.data.frame(nanoparquet::read_parquet(data_file)), exchange))
})
# Anchor OOB extraction to the original data-only returnOOB attribute using the same artificial data and seed.
toy_original <- missRanger::missRanger(toy_dev, formula = stats::as.formula(". ~ ."), pmm.k = small$pmm_k, num.trees = small$num_trees, maxiter = small$maxiter, num.threads = threads, seed = 2125L, returnOOB = TRUE, respect.unordered.factors = "order", verbose = 0L)
toy_oob <- oob_report(toy_fit, 1L)
stopifnot(isTRUE(all.equal(toy_oob$final$OOB, as.numeric(attr(toy_original, "oob")[toy_oob$final$Variable]), tolerance = 0)))
show_table(toy_oob$final, "Synthetic OOB extraction control - not study results")
cat("Final-iteration OOB agrees with the original returnOOB attribute on the synthetic control.\n")
rm(toy, toy_dev, toy_new, toy_fit, toy_prediction, toy_original, toy_oob, bad)
cat("Synthetic control passed. These are not study results.\n")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Missing value imputation by random forests




Variables to impute:		a, b
Variables used to impute:	a, b, complete

iter 1 

  |                                                                            
  |                                                                      |   0%


  |                                                                            
  |===================================                                   |  50%


  |                                                                            
  |======================================================================| 100%

Missing value imputation by random forests



...


  |                                                                            
  |======================================================================| 100%

Imputation,Variable,Scale,OOB,Selected_iteration,Missing_before,Above_threshold
1,a,Normalized MSE (1 - R^2),1.1288,1,10,TRUE
1,b,Misclassification proportion,0.8810,1,12,TRUE


Final-iteration OOB agrees with the original returnOOB attribute on the synthetic control.


Synthetic control passed. These are not study results.


Elapsed: 0.003 minutes


## 3. Raw data and frozen participant split


In [5]:
#| label: prep-load-raw-and-identity
#| message: false
.t0 <- proc.time()
# Load the saved workspace privately; retain only verified inputs and the identity bridge.
required_sources <- c("raw_workspace", "centers", "split", "schema_reference")
input_paths <- setNames(vapply(required_sources, source_path, character(1L)), required_sources)
if (!all(file.exists(input_paths))) stop("Missing configured preparation input.", call. = FALSE)
input_hashes <- vapply(input_paths, function(p) digest::digest(file = p, algo = "sha256"), character(1L))
raw_env <- new.env(parent = baseenv())
invisible(load(input_paths[["raw_workspace"]], envir = raw_env))
raw_name <- config$sources$legacy_raw_object
identity_name <- config$sources$raw_identity_object
stopifnot(exists(raw_name, raw_env, inherits = FALSE), exists(identity_name, raw_env, inherits = FALSE))
raw_all <- as.data.frame(get(raw_name, raw_env, inherits = FALSE))
model <- get(identity_name, raw_env, inherits = FALSE)
stopifnot(all(c("original_row_id", "hash_key", names(raw_all)) %in% names(model)))
# Factor metadata may differ; require identical values and missingness for every row.
same_raw <- vapply(names(raw_all), function(v) identical(is.na(raw_all[[v]]), is.na(model[[v]])) && identical(as.character(raw_all[[v]]), as.character(model[[v]])), logical(1L))
if (!all(same_raw)) stop("Raw/model identity bridge differs in: ", paste(names(same_raw)[!same_raw], collapse = ", "), call. = FALSE)
identity_all <- as.data.frame(model[, c("original_row_id", "hash_key")])
stopifnot(identical(as.integer(identity_all$original_row_id), seq_len(nrow(raw_all))), !anyNA(identity_all$hash_key), !anyDuplicated(identity_all$hash_key))
inherited_exclusions <- if (exists("high_uncertainty_imps", raw_env, inherits = FALSE)) get("high_uncertainty_imps", raw_env, inherits = FALSE) else character()
centers_all <- readRDS(input_paths[["centers"]])
stopifnot(length(centers_all) == nrow(raw_all), all(outcome_names %in% names(raw_all)), !anyNA(raw_all[outcome_names]))
audit_columns <- c("original_row_id", "first_disch", "death_date", "adm_disch_reason", "tr_outcome", "adm_age_rec3", "sex_rec", "primary_sub_mod", "any_phys_dx", "death_time_from_disch_m")
stopifnot(all(audit_columns %in% names(model)))
# Retain a separate descriptive audit frame; it never enters an imputer or a model.
discharge_audit <- as.data.frame(model)[, audit_columns, drop = FALSE]
rm(model, raw_env)
invisible(gc())
# Convert binary outcomes using their observed coding, without using them as predictors.
to01 <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  value <- suppressWarnings(as.integer(x))
  stopifnot(!anyNA(value))
  if (all(value %in% c(0L, 1L))) return(value)
  if (all(value %in% c(1L, 2L))) return(value - 1L)
  stop("Unexpected event coding.", call. = FALSE)
}
death_event <- to01(raw_all$death_event)
drop_artifact <- grepl("adm reasons", as.character(raw_all$tr_outcome), ignore.case = TRUE) & death_event == 1L & as.numeric(raw_all$death_time_from_disch_m) <= 0.23
drop_other <- grepl("other", as.character(raw_all$tr_outcome), ignore.case = TRUE)
keep <- !(drop_artifact | drop_other)
stopifnot(!anyNA(keep))
discharge_audit$death_event <- death_event
discharge_audit$drop_artifact <- drop_artifact
discharge_audit$drop_other <- drop_other
raw <- raw_all[keep, , drop = FALSE]
identity <- identity_all[keep, , drop = FALSE]
center_id <- centers_all[keep]
rownames(raw) <- rownames(identity) <- NULL
show_table(data.frame(Stage = c("Before historical eligibility filters", "Administrative death artifact", "Other treatment outcome", "Eligible"), N = c(nrow(raw_all), sum(drop_artifact), sum(drop_other), sum(keep))), "Historical cohort eligibility")
rm(raw_all, identity_all, centers_all, death_event, drop_artifact, drop_other, keep)
invisible(gc())
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Stage,N
Before historical eligibility filters,88504
Administrative death artifact,137
Other treatment outcome,215
Eligible,88152


Elapsed: 0.483 minutes


In [6]:
#| label: prep-verify-historical-split
#| message: false
.t0 <- proc.time()
# Reuse the frozen split only after checking its complete positional lineage.
split <- as.data.frame(nanoparquet::read_parquet(input_paths[["split"]]))
reference <- as.data.frame(nanoparquet::read_parquet(input_paths[["schema_reference"]]))
stopifnot(nrow(raw) == nrow(split), nrow(raw) == nrow(reference), identical(as.integer(split$row_id), seq_len(nrow(raw))))
stopifnot(!anyNA(split$is_train), all(split$is_train %in% c(TRUE, FALSE)))
is_train <- as.logical(split$is_train)
stopifnot(any(is_train), any(!is_train))
same_values <- function(a, b, tolerance = 1e-8) {
  if (length(a) != length(b) || !identical(is.na(a), is.na(b))) return(FALSE)
  observed <- !is.na(a)
  if (is.numeric(a) && is.numeric(b)) return(all(abs(a[observed] - b[observed]) <= tolerance))
  identical(as.character(a[observed]), as.character(b[observed]))
}
stopifnot(same_values(round(as.numeric(raw$death_time_from_disch_m), 2), as.numeric(split$death_time_from_disch_m)), same_values(raw$adm_age_rec3, split$adm_age_rec3))
# Use the historical parquet only as an alignment control, never to learn a new schema.
predictor_names <- setdiff(names(raw), outcome_names)
assert_predictors(raw[predictor_names])
common <- intersect(predictor_names, names(reference))
observed_match <- vapply(common, function(v) {
  observed <- !is.na(raw[[v]])
  same_values(raw[[v]][observed], reference[[v]][observed])
}, logical(1L))
if (!all(observed_match)) stop("Observed values differ from the historical reference: ", paste(names(observed_match)[!observed_match], collapse = ", "), call. = FALSE)
stopifnot(same_values(as.character(center_id), as.character(reference$center_id)))
# Restore endpoints separately using the verified August competing-death convention.
outcomes <- raw[outcome_names]
outcomes$death_event <- to01(outcomes$death_event)
outcomes$readmit_event <- to01(outcomes$readmit_event)
competing <- outcomes$death_event == 1L & outcomes$death_time_from_disch_m <= outcomes$readmit_time_from_disch_m
outcomes$readmit_time_from_disch_m[competing] <- outcomes$death_time_from_disch_m[competing]
outcomes$readmit_event[competing] <- 0L
endpoint_common <- intersect(outcome_names, names(reference))
endpoint_match <- vapply(endpoint_common, function(v) same_values(outcomes[[v]], reference[[v]]), logical(1L))
if (!all(endpoint_match)) stop("Endpoint convention differs from the historical reference: ", paste(names(endpoint_match)[!endpoint_match], collapse = ", "), call. = FALSE)
identity$legacy_row_id <- as.integer(split$row_id)
identity$is_train <- is_train
identity$center_id <- center_id
stopifnot(!length(intersect(identity$hash_key[is_train], identity$hash_key[!is_train])))
show_table(data.frame(Check = c("Raw/model identity bridge", "All common observed predictors", "Center ordering", "Restored endpoints", "Participant overlap"), Passed = TRUE), "Row identity and split checks")
show_table(data.frame(Partition = c("Development", "Holdout"), Rows = c(sum(is_train), sum(!is_train))), "Frozen participant split")
rm(reference, split)
invisible(gc())
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Check,Passed
Raw/model identity bridge,TRUE
All common observed predictors,TRUE
Center ordering,TRUE
Restored endpoints,TRUE
Participant overlap,TRUE


Partition,Rows
Development,70521
Holdout,17631


Elapsed: 0.007 minutes


### Early recorded deaths: audit of the unchanged eligibility rule
This section describes the original preparation cohort, using its processed discharge and death dates. A recorded death is not treated as false: the uncertainty concerns whether the discharge represents a prospective time origin or an administrative closure. Dates may already include historical corrections, so their ordering cannot establish the original chronology.
The historical administrative rule remains unchanged. Calendar days 1-7 are shown separately from the cutoff of 0.23 calendar months. Excluded participants precede the historical split and are not assigned retrospectively to development or holdout. Retained covariate comparisons use development only; cohort counts and the explicitly labelled observed-mortality sensitivity inspect outcomes across the whole historical cohort. These diagnostics must not be used to choose exclusions, predictors, thresholds or models.


In [7]:
#| label: prep-early-death-counts
#| message: false
.t0 <- proc.time()
# Audit processed dates separately from predictor preparation; preserve all historical filters.
stopifnot(inherits(discharge_audit$first_disch, "Date"), inherits(discharge_audit$death_date, "Date"))
discharge_audit$Days_after_discharge <- as.numeric(discharge_audit$death_date - discharge_audit$first_disch)
discharge_audit$Calendar_week <- discharge_audit$death_event == 1L & !is.na(discharge_audit$Days_after_discharge) & discharge_audit$Days_after_discharge >= 1 & discharge_audit$Days_after_discharge <= 7
discharge_audit$Positive_month_window <- discharge_audit$death_event == 1L & discharge_audit$death_time_from_disch_m > 0 & discharge_audit$death_time_from_disch_m <= 0.23
discharge_audit$Disposition <- ifelse(discharge_audit$drop_artifact, "Excluded: administrative rule", ifelse(discharge_audit$drop_other, "Excluded: other outcome", "Retained"))
audit_match <- match(identity$original_row_id, discharge_audit$original_row_id)
stopifnot(!anyNA(audit_match), !anyDuplicated(audit_match), identical(discharge_audit$original_row_id[audit_match], identity$original_row_id))
discharge_audit$Disposition[audit_match] <- ifelse(is_train, "Retained: development", "Retained: holdout")
stopifnot(sum(discharge_audit$Disposition == "Retained: development") == sum(is_train), sum(discharge_audit$Disposition == "Retained: holdout") == sum(!is_train))
audit_days <- discharge_audit$Days_after_discharge
audit_period <- ifelse(discharge_audit$death_event == 0L, "No death recorded during follow-up", ifelse(is.na(audit_days), "Death: missing dates", ifelse(audit_days < 0, "Death: before processed discharge", ifelse(audit_days == 0, "Death: same processed date", ifelse(audit_days <= 7, "Death: days 1-7", ifelse(audit_days <= 30, "Death: days 8-30", "Death: after day 30"))))))
audit_counts <- as.data.frame.matrix(table(audit_period, discharge_audit$Disposition))
audit_counts <- data.frame(Recorded_timing = rownames(audit_counts), audit_counts, row.names = NULL, check.names = FALSE)
show_table(audit_counts, "Historical cohort: recorded death timing and unchanged disposition")
# A calendar week and a fractional calendar month are different definitions.
audit_boundary <- data.frame(Definition = c("Calendar days 1-7", "Positive time <=0.23 months", "Days 1-7 but outside positive 0.23-month window"), Recorded_deaths = c(sum(discharge_audit$Calendar_week), sum(discharge_audit$Positive_month_window), sum(discharge_audit$Calendar_week & !discharge_audit$Positive_month_window)), Removed_by_administrative_rule = c(sum(discharge_audit$Calendar_week & discharge_audit$drop_artifact), sum(discharge_audit$Positive_month_window & discharge_audit$drop_artifact), sum(discharge_audit$Calendar_week & !discharge_audit$Positive_month_window & discharge_audit$drop_artifact)))
show_table(audit_boundary, "Exact calendar week versus the unchanged historical month cutoff")
audit_week_rows <- discharge_audit[discharge_audit$Calendar_week, , drop = FALSE]
audit_reasons <- as.data.frame(table(Disposition = audit_week_rows$Disposition, Treatment_outcome = as.character(audit_week_rows$tr_outcome), Administrative_subreason = ifelse(is.na(audit_week_rows$adm_disch_reason), "(missing)", as.character(audit_week_rows$adm_disch_reason))), stringsAsFactors = FALSE)
audit_reasons <- audit_reasons[audit_reasons$Freq > 0L, , drop = FALSE]
show_table(audit_reasons, "Recorded discharge reasons among deaths on calendar days 1-7")
cat("Counts inspect outcomes across the historical cohort; they do not redefine eligibility or select predictors.\n")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Recorded_timing,Excluded: administrative rule,Excluded: other outcome,Retained: development,Retained: holdout
Death: after day 30,0,7,2817,710
Death: days 1-7,15,0,23,3
Death: days 8-30,0,0,39,8
Death: same processed date,122,4,160,39
No death recorded during follow-up,0,204,67482,16871


Definition,Recorded_deaths,Removed_by_administrative_rule
Calendar days 1-7,41,15
Positive time <=0.23 months,39,15
Days 1-7 but outside positive 0.23-month window,2,0


Disposition,Treatment_outcome,Administrative_subreason,Freq
Retained: development,adm discharge - rule violation/undet,(missing),8
Retained: holdout,adm discharge - rule violation/undet,(missing),3
Retained: development,completion,(missing),3
Retained: development,dropout,(missing),9
Retained: development,referral,(missing),2
Excluded: administrative rule,adm discharge - adm reasons,death,13
Excluded: administrative rule,adm discharge - adm reasons,no_local_service,2
Retained: development,adm discharge - rule violation/undet,rule_violation,1


Counts inspect outcomes across the historical cohort; they do not redefine eligibility or select predictors.


Elapsed: 0.002 minutes


In [8]:
#| label: prep-early-death-profiles
#| message: false
.t0 <- proc.time()
# Compare observed covariates descriptively; retained comparators come only from development.
audit_groups <- list("Excluded before split: administrative, days 1-7" = discharge_audit$drop_artifact & !discharge_audit$drop_other & discharge_audit$Calendar_week, "Retained development: days 1-7" = discharge_audit$Disposition == "Retained: development" & discharge_audit$Calendar_week, "Retained development: deaths after day 7" = discharge_audit$Disposition == "Retained: development" & discharge_audit$death_event == 1L & !is.na(discharge_audit$Days_after_discharge) & discharge_audit$Days_after_discharge > 7)
audit_numeric <- function(x) {
  observed <- x[!is.na(x)]
  if (!length(observed)) return(sprintf("Not observed; missing=%d", length(x)))
  q <- stats::quantile(observed, c(0.25, 0.5, 0.75), names = FALSE)
  sprintf("%.1f [%.1f, %.1f]; observed=%d; missing=%d", q[2], q[1], q[3], length(observed), sum(is.na(x)))
}
audit_proportion <- function(x, level) {
  observed <- x[!is.na(x)]
  if (!length(observed)) return(sprintf("Not observed; missing=%d", length(x)))
  n <- sum(as.character(observed) == level)
  sprintf("%d/%d (%.1f%%); missing=%d", n, length(observed), 100 * n / length(observed), sum(is.na(x)))
}
audit_substances <- sort(unique(as.character(discharge_audit$primary_sub_mod[Reduce(function(a, b) a | b, audit_groups) & !is.na(discharge_audit$primary_sub_mod)])))
audit_profile <- data.frame(Characteristic = c("Participants", "Admission age: median [IQR]", "Women", "Physical diagnosis recorded", paste("Primary substance:", audit_substances)), check.names = FALSE)
for (group in names(audit_groups)) {
  d <- discharge_audit[audit_groups[[group]], , drop = FALSE]
  audit_profile[[group]] <- c(nrow(d), audit_numeric(d$adm_age_rec3), audit_proportion(d$sex_rec, "woman"), audit_proportion(d$any_phys_dx, "TRUE"), vapply(audit_substances, function(level) audit_proportion(d$primary_sub_mod, level), character(1L)))
}
show_table(audit_profile, "Observed profiles only: excluded early deaths and retained development deaths")
cat("All groups have recorded deaths. These descriptions cannot adjudicate whether a discharge date is correct.\n")
cat("No missing values were imputed, no hypothesis tests were added, and no holdout covariate profiles were inspected.\n")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Characteristic,"Excluded before split: administrative, days 1-7",Retained development: days 1-7,Retained development: deaths after day 7
Participants,15,23,2856
Admission age: median [IQR],"48.1 [44.4, 56.6]; observed=15; missing=0","38.8 [31.9, 47.1]; observed=23; missing=0","43.4 [34.5, 51.2]; observed=2856; missing=0"
Women,3/15 (20.0%); missing=0,3/23 (13.0%); missing=0,619/2856 (21.7%); missing=0
Physical diagnosis recorded,4/15 (26.7%); missing=0,6/23 (26.1%); missing=0,563/2847 (19.8%); missing=9
Primary substance: alcohol,11/15 (73.3%); missing=0,9/23 (39.1%); missing=0,1664/2856 (58.3%); missing=0
Primary substance: cocaine paste,0/15 (0.0%); missing=0,11/23 (47.8%); missing=0,759/2856 (26.6%); missing=0
Primary substance: cocaine powder,4/15 (26.7%); missing=0,2/23 (8.7%); missing=0,279/2856 (9.8%); missing=0
Primary substance: marijuana,0/15 (0.0%); missing=0,0/23 (0.0%); missing=0,101/2856 (3.5%); missing=0
Primary substance: others,0/15 (0.0%); missing=0,1/23 (4.3%); missing=0,53/2856 (1.9%); missing=0


All groups have recorded deaths. These descriptions cannot adjudicate whether a discharge date is correct.


No missing values were imputed, no hypothesis tests were added, and no holdout covariate profiles were inspected.


Elapsed: 0.001 minutes


In [9]:
#| label: prep-early-death-observed-risk-sensitivity
#| message: false
.t0 <- proc.time()
# Isolate the descriptive consequence of reinstating positive-time first-week administrative deaths.
stopifnot(requireNamespace("survival", quietly = TRUE), utils::packageVersion("survival") == base::package_version(lock$Packages$survival$Version))
audit_primary <- !(discharge_audit$drop_artifact | discharge_audit$drop_other)
audit_reinstate <- discharge_audit$drop_artifact & !discharge_audit$drop_other & discharge_audit$Calendar_week
stopifnot(!any(audit_primary & audit_reinstate), all(discharge_audit$death_time_from_disch_m[audit_reinstate] > 0))
audit_horizons <- c(6, 12, 36, 60)
audit_km <- function(rows) {
  d <- discharge_audit[rows, c("death_time_from_disch_m", "death_event"), drop = FALSE]
  fit <- survival::survfit(survival::Surv(death_time_from_disch_m, death_event) ~ 1, data = d)
  out <- summary(fit, times = audit_horizons, extend = FALSE)
  stopifnot(length(out$surv) == length(audit_horizons), all(out$n.risk > 0))
  list(n = nrow(d), deaths = sum(d$death_event), risk = 1 - as.numeric(out$surv), at_risk = out$n.risk)
}
audit_km_primary <- audit_km(audit_primary)
audit_km_restored <- audit_km(audit_primary | audit_reinstate)
stopifnot(audit_km_primary$n == nrow(raw), audit_km_restored$n - audit_km_primary$n == sum(audit_reinstate), audit_km_restored$deaths - audit_km_primary$deaths == sum(audit_reinstate), all(audit_km_restored$risk >= audit_km_primary$risk - 1e-12))
audit_sensitivity <- data.frame(Horizon_months = audit_horizons, Current_rule_N = audit_km_primary$n, Reinstated_N = sum(audit_reinstate), Current_rule_KM_percent = 100 * audit_km_primary$risk, With_early_cases_KM_percent = 100 * audit_km_restored$risk, Difference_percentage_points = 100 * (audit_km_restored$risk - audit_km_primary$risk), Current_rule_at_risk = audit_km_primary$at_risk)
show_table(audit_sensitivity, "Descriptive observed-mortality sensitivity: whole historical cohort, not model performance")
cat("This diagnostic reads outcomes across development and holdout. The already fixed rule, split and models are unchanged.\n")
cat("Only excluded administrative deaths on calendar days 1-7 are reinstated hypothetically; zero-time, pre-discharge and other exclusions are not reinstated.\n")
cat("No predictive model, tuning, imputation, bootstrap or predictive-performance metric is fitted or selected here.\n")
rm(discharge_audit, audit_week_rows, d)
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Horizon_months,Current_rule_N,Reinstated_N,Current_rule_KM_percent,With_early_cases_KM_percent,Difference_percentage_points,Current_rule_at_risk
6,88152,15,0.6688,0.6867,0.0179,80104
12,88152,15,1.0410,1.0588,0.0178,77085
36,88152,15,2.8532,2.8707,0.0175,58816
60,88152,15,4.4867,4.5039,0.0172,39615


This diagnostic reads outcomes across development and holdout. The already fixed rule, split and models are unchanged.


Only excluded administrative deaths on calendar days 1-7 are reinstated hypothetically; zero-time, pre-discharge and other exclusions are not reinstated.


No predictive model, tuning, imputation, bootstrap or predictive-performance metric is fitted or selected here.


Elapsed: 0.003 minutes


**Interpretation and limitation.** Similarity in age, sex, substance use or recorded physical diagnoses cannot establish that an early death followed a genuine discharge. Administrative subreasons provide supporting evidence, not adjudication. The before/after Kaplan-Meier comparison measures the descriptive consequence of a cohort definition on observed mortality; it does not measure changes in discrimination, calibration or any individual's prediction. No inferential test is added for the potentially small excluded subgroup.
The primary preparation and historical participant split remain unchanged. This audit's access to holdout outcomes is disclosed, so the complete workflow is not described as wholly independent of the holdout. Any later assessment of predictive influence would require verified frozen predictions for the excluded cases and a separate checkpoint.


**Executed findings (2026-10-02).** The processed dates identify 41 deaths on calendar days 1-7: 15 excluded by the unchanged administrative rule and 26 retained (23 development, 3 holdout). Thirteen of the 15 excluded cases have administrative subreason `death`; two have `no_local_service`. The exact calendar week contains two more deaths than the positive 0.23-month window, but neither is excluded by the administrative rule.

The excluded first-week group is descriptively older (median admission age 48.1 years) than the retained development first-week group (38.8) and later-death group (43.4); alcohol is the primary substance in 11/15, 9/23 and 1664/2856, respectively. These small, selected groups do not establish exchangeability or the validity of a discharge date. Hypothetically reinstating the 15 positive-time cases increases observed Kaplan-Meier mortality by 0.0179, 0.0178, 0.0175 and 0.0172 percentage points at 6, 12, 36 and 60 months. This is not an assessment of predictive performance.

The historical administrative exclusion also removes 122 deaths recorded on the same processed date as discharge; 199 same-date deaths remain in the final cohort. They are counted separately and are not reinstated in this first-week sensitivity. Processed dates and the current rule therefore do not establish that every discharge was a genuine, prospectively observed time origin.


## 4. Development predictor contract


In [10]:
#| label: prep-freeze-development-contract
#| message: false
.t0 <- proc.time()
# Learn factor levels from development; preserve the study-defined ordered level sequence.
development <- raw[is_train, predictor_names, drop = FALSE]
holdout <- raw[!is_train, predictor_names, drop = FALSE]
for (v in predictor_names) {
  if (is.factor(development[[v]])) {
    development[[v]] <- droplevels(development[[v]])
    unseen <- setdiff(unique(as.character(holdout[[v]][!is.na(holdout[[v]])])), levels(development[[v]]))
    if (length(unseen)) stop("Holdout category absent from development: ", v, call. = FALSE)
    holdout[[v]] <- factor(as.character(holdout[[v]]), levels = levels(development[[v]]), ordered = is.ordered(development[[v]]))
  }
}
assert_predictors(development)
unsupported <- predictor_names[colSums(is.na(development)) == 0L & colSums(is.na(holdout)) > 0L]
if (length(unsupported)) stop("Holdout has missing values without a development imputation target: ", paste(unsupported, collapse = ", "), call. = FALSE)
contract <- lapply(development, function(x) list(class = class(x), levels = if (is.factor(x)) levels(x) else NULL))
audit <- data.frame(predictor = predictor_names, development_missing = colSums(is.na(development)), development_unique = vapply(development, function(x) length(unique(x[!is.na(x)])), integer(1L)), historically_excluded_after_imputation = predictor_names %in% inherited_exclusions, row.names = NULL)
show_table(transform(audit, development_missing_percent = 100 * development_missing / nrow(development)), "Development missingness and historical quality exclusions")
cat("Historical quality-based exclusions are audit flags, not automatic exclusions.\n")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


predictor,development_missing,development_unique,historically_excluded_after_imputation,development_missing_percent
adm_age_rec3,0,4564,FALSE,0.0000
porc_pobr,0,863,FALSE,0.0000
dit_m,0,11115,FALSE,0.0000
num_trat_ant,4767,12,TRUE,6.7597
age_subs_onset_rec2,3441,57,TRUE,4.8794
sex_rec,2,2,FALSE,0.0028
tenure_status_household,3607,5,FALSE,5.1148
cohabitation,0,4,FALSE,0.0000
sub_dep_icd10_status,0,2,FALSE,0.0000
any_violence,14106,2,FALSE,20.0026


Historical quality-based exclusions are audit flags, not automatic exclusions.


Elapsed: 0.003 minutes


In [11]:
#| label: prep-save-raw-partitions
#| message: false
.t0 <- proc.time()
# Preserve raw partitions for provenance; the approved ML workflow reuses full-development imputations.
provenance <- list(format_version = 1L, input_sha256 = input_hashes, candidate_scope = config$preparation$candidate_scope, predictor_contract = contract, settings = config$imputation, r_version = as.character(getRversion()), package_versions = vapply(packages, function(p) as.character(utils::packageVersion(p)), character(1L)), eligibility = "Historical other/admin-death<=0.23 rule", endpoint_rule = "August competing death <= readmission")
fingerprint <- substr(digest::digest(provenance, algo = "sha256"), 1L, 12L)
dated_id <- paste(format(Sys.Date(), "%Y-%m-%d"), fingerprint, sep = "_")
prepared <- list(provenance = provenance, identity = identity, development = development, holdout = holdout, outcomes = outcomes, raw_outcomes = raw[outcome_names], audit = audit)
# Keep the creation date across sessions so matching models are never refitted just because the date changed.
existing <- list.files(object_dir, pattern = paste0("^preparation_raw_[0-9-]+_", fingerprint, "\\.rds$"), full.names = TRUE)
if (length(existing) > 1L) stop("Multiple raw-partition caches have the same fingerprint.", call. = FALSE)
prepared_path <- if (length(existing)) existing[[1L]] else file.path(object_dir, paste0("preparation_raw_", dated_id, ".rds"))
dated_id <- sub("\\.rds$", "", sub("^preparation_raw_", "", basename(prepared_path)))
if (file.exists(prepared_path)) {
  saved <- readRDS(prepared_path)
  check_cache(saved$provenance, provenance)
  stopifnot(identical(saved, prepared))
  rm(saved)
  cat("Reused verified raw-partition cache.\n")
} else {
  saveRDS(prepared, prepared_path, compress = FALSE, version = 3L)
  stopifnot(identical(prepared, readRDS(prepared_path)))
  written <- c(written, prepared_path)
}
show_table(data.frame(Artifact = substring(prepared_path, nchar(root) + 2L), Status = if (prepared_path %in% written) "written" else "reused", Bytes = as.numeric(file.info(prepared_path)$size)), "Raw partition artifact")
cat("Raw partitions saved; no imputer has seen participant outcomes.\n")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Reused verified raw-partition cache.


Artifact,Status,Bytes
outputs/objects/preparation_raw_2026-10-02_71f0ce969d33.rds,reused,31048356


Raw partitions saved; no imputer has seen participant outcomes.


Elapsed: 0.006 minutes


## 5. Imputation and diagnostics
`preflight` skips cohort fitting. An optional `pilot` uses 5,000 development rows, 20 trees and two iterations only to measure computing cost; it supplies no article estimates and never reads holdout outcomes. After the resource checkpoint, `first_imputation` fits and saves one complete imputation; `full` completes the configured set sequentially, reusing matching files.
No runtime estimate is inferred from the synthetic control. Fitted models retain the development data required by predictive mean matching. They must remain private research artifacts.
Performance comparisons against the manuscript are pending; this notebook prepares inputs only.


In [12]:
#| label: prep-fit-and-export-frozen-imputers
#| message: false
.t0 <- proc.time()
# Fit sequentially only after the explicitly selected compute mode permits cohort work.
oob_runs <- list()
oob_histories <- list()
missing_runs <- list()
if (config$preparation$mode == "pilot") {
  # This optional timing-only pilot produces no imputations or performance estimates for the article.
  set.seed(as.integer(config$imputation$seed))
  pilot_rows <- sort(sample.int(nrow(development), min(5000L, nrow(development))))
  pilot_settings <- utils::modifyList(config$imputation, list(num_trees = 20L, maxiter = 2L))
  pilot_fit <- fit_imputer(development[pilot_rows, , drop = FALSE], pilot_settings, as.integer(config$imputation$seed))
  show_table(data.frame(Purpose = "Timing only; not reportable", Development_rows = length(pilot_rows), Trees = pilot_settings$num_trees, Max_iterations = pilot_settings$maxiter, Threads = threads, Elapsed_minutes = unname((proc.time() - .t0)["elapsed"]) / 60, Fitted_object_MiB = as.numeric(object.size(pilot_fit)) / 1024^2), "Optional timing pilot")
  rm(pilot_fit)
  invisible(gc())
} else if (config$preparation$mode == "preflight") {
  cat("Cohort fitting skipped: awaiting the user's compute decision.\n")
} else {
  n_imputations <- if (config$preparation$mode == "first_imputation") 1L else as.integer(config$imputation$n_imputations)
  for (imp in seq_len(n_imputations)) {
    imp_started <- proc.time()
    cat("Starting complete-development imputation", imp, "of", n_imputations, "at", format(Sys.time()), "\n")
    flush.console()
    seed <- as.integer(config$imputation$seed) + imp - 1L
    imputation_meta <- list(provenance = provenance, imputation = imp, seed = seed)
    model_path <- file.path(object_dir, paste0("imputer_imp", imp, "_", dated_id, ".rds"))
    data_path <- file.path(object_dir, paste0("completed_imp", imp, "_", dated_id, ".parquet"))
    if (file.exists(model_path)) {
      cached <- readRDS(model_path)
      check_cache(cached$metadata, imputation_meta)
      fit <- cached$fit
      rm(cached)
      cat("Reused verified imputer", imp, "\n")
    } else {
      fit <- fit_imputer(development, config$imputation, seed)
      assert_observed(development, fit$data)
      stopifnot(identical(names(fit$data_raw), predictor_names))
      saveRDS(list(metadata = imputation_meta, fit = fit), model_path, compress = FALSE, version = 3L)
      written <- c(written, model_path)
      # Verify persistence using the saved model, without holding two copies of its forests.
      rm(fit)
      invisible(gc())
      cached <- readRDS(model_path)
      check_cache(cached$metadata, imputation_meta)
      fit <- cached$fit
      rm(cached)
    }
    # Require the exact complete development training data, including its missingness.
    stopifnot(identical(fit$data_raw, development))
    # Retain lightweight OOB diagnostics before releasing each fitted imputer.
    diagnostics <- oob_report(fit, imp)
    oob_runs[[imp]] <- diagnostics$final
    oob_histories[[imp]] <- diagnostics$history
    # Apply development forests and donors; append outcomes and identifiers only afterward.
    if (file.exists(data_path)) {
      completed_cache <- as.data.frame(nanoparquet::read_parquet(data_path))
      expected_metadata <- cbind(identity, outcomes)
      expected_metadata[] <- lapply(expected_metadata, function(x) if (is.factor(x)) as.character(x) else x)
      stopifnot(identical(completed_cache[names(expected_metadata)], expected_metadata))
      assert_observed(raw[predictor_names], completed_cache[predictor_names])
      # Verify that completed development values match this saved imputer, not another run.
      stopifnot(all(vapply(predictor_names, function(v) identical(as.character(completed_cache[[v]][is_train]), as.character(fit$data[[v]])), logical(1L))))
      missing_runs[[imp]] <- rbind(missing_report(development, completed_cache[is_train, predictor_names, drop = FALSE], imp, "Development"), missing_report(holdout, completed_cache[!is_train, predictor_names, drop = FALSE], imp, "Holdout"))
      cat("Reused completed data for imputation", imp, "\n")
      rm(fit, completed_cache, expected_metadata)
      invisible(gc())
      next
    }
    completed_holdout <- apply_imputer(fit, holdout, config$imputation, seed)
    missing_runs[[imp]] <- rbind(missing_report(development, fit$data, imp, "Development"), missing_report(holdout, completed_holdout, imp, "Holdout"))
    completed <- raw[predictor_names]
    completed[is_train, ] <- fit$data
    completed[!is_train, ] <- completed_holdout
    assert_observed(raw[predictor_names], completed)
    export <- cbind(identity, outcomes, completed)
    export[] <- lapply(export, function(x) if (is.factor(x)) as.character(x) else x)
    nanoparquet::write_parquet(export, data_path)
    stopifnot(identical(as.data.frame(nanoparquet::read_parquet(data_path)), export))
    written <- c(written, data_path)
    show_table(data.frame(Imputation = imp, Elapsed_minutes = unname((proc.time() - imp_started)["elapsed"]) / 60, Fitted_object_MiB = as.numeric(object.size(fit)) / 1024^2), "Completed imputation resource use")
    rm(fit, completed_holdout, completed, export)
    invisible(gc())
  }
}
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Starting complete-development imputation 1 of 5 at 2026-10-03 12:11:02 


Reused verified imputer 1 
Reused completed data for imputation 1 
Starting complete-development imputation 2 of 5 at 2026-10-03 12:11:24 


Reused verified imputer 2 
Reused completed data for imputation 2 
Starting complete-development imputation 3 of 5 at 2026-10-03 12:11:43 


Reused verified imputer 3 
Reused completed data for imputation 3 
Starting complete-development imputation 4 of 5 at 2026-10-03 12:12:02 


Reused verified imputer 4 
Reused completed data for imputation 4 
Starting complete-development imputation 5 of 5 at 2026-10-03 12:12:21 


Reused verified imputer 5 
Reused completed data for imputation 5 


Elapsed: 1.608 minutes


### OOB and missingness diagnostics
The original report summarizes the final OOB error across imputations: mean, SD, minimum and maximum. For regression, missRanger reports normalized MSE (`1 - R^2`), which can exceed one; for classification it reports the proportion misclassified. These are not NRMSE and must not be pooled across fitting iterations.
Confirmed on 2026-10-02: the exclusion-review rule is mean final OOB across the five imputations greater than 0.70. `Runs_above_threshold` reports how many of the five imputations exceed 0.70 as supporting information; it is not a separate exclusion rule. The original code subsequently removed three named predictors. No new exclusion is applied automatically. Any changed mean flag triggers review before ML.
Cohort OOB and missingness tables appear here after fitting. Synthetic diagnostics above test the extraction code only.


In [13]:
#| label: prep-oob-and-missingness-diagnostics
#| message: false
.t0 <- proc.time()
# Reproduce the original per-variable OOB report without averaging over fitting iterations.
if (!length(oob_runs)) {
  show_table(data.frame(Diagnostic = c("Variable by imputation OOB", "Mean, SD, minimum and maximum OOB", "Iteration history and selected stopping point", "Historical exclusion review", "Missingness before and after"), Status = "Pending complete-development imputation"), "Official cohort diagnostics - pending execution")
} else {
  oob_values <- do.call(rbind, oob_runs)
  oob_history <- do.call(rbind, oob_histories)
  missing_values <- do.call(rbind, missing_runs)
  threshold <- config$imputation_diagnostics$oob_threshold
  oob_summary <- do.call(rbind, lapply(split(oob_values, oob_values$Variable), function(x) data.frame(Variable = x$Variable[[1L]], Scale = x$Scale[[1L]], Imputations = nrow(x), Mean_OOB = mean(x$OOB), SD_OOB = stats::sd(x$OOB), Best_Run = min(x$OOB), Worst_Run = max(x$OOB), Runs_above_threshold = sum(x$OOB > threshold), Mean_above_threshold = mean(x$OOB) > threshold, Historically_excluded = x$Variable[[1L]] %in% inherited_exclusions)))
  oob_summary <- oob_summary[order(oob_summary$Mean_OOB), , drop = FALSE]
  decision <- oob_summary[, c("Variable", "Imputations", "Mean_OOB", "Runs_above_threshold", "Historically_excluded", "Mean_above_threshold")]
  decision$Review_changed_flag <- decision$Historically_excluded != decision$Mean_above_threshold
  show_table(oob_values, "Final OOB by predictor and imputation")
  show_table(oob_summary, "Original OOB summary: mean, SD, best and worst run")
  show_table(oob_history, "OOB fitting history and selected iteration")
  show_table(decision, "Historical exclusion review - no automatic variable removal")
  show_table(missing_values, "Missingness before and after frozen imputation")
  stopifnot(all(missing_values$Missing_after == 0L))
  if (length(oob_runs) < config$imputation$n_imputations) {
    cat("Partial run: the across-imputation exclusion review is not complete.\n")
  } else if (any(decision$Review_changed_flag)) {
    stop("Substantial review required: development-only OOB flags differ from historical exclusions. No selection has been applied; discuss the displayed table before notebook 01.", call. = FALSE)
  } else {
    cat("Mean-OOB flags agree with historical exclusions. Review the number of runs above threshold at the notebook checkpoint; no variables were removed here.\n")
  }
}
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Imputation,Variable,Scale,OOB,Selected_iteration,Missing_before,Above_threshold
1,sex_rec,Misclassification proportion,0.1229,2,2,FALSE
1,evaluacindelprocesoteraputico,Misclassification proportion,0.0987,2,3,FALSE
1,eva_consumo,Misclassification proportion,0.1496,2,17,FALSE
1,eva_fam,Misclassification proportion,0.1875,2,17,FALSE
1,eva_relinterp,Misclassification proportion,0.1649,2,17,FALSE
1,eva_ocupacion,Misclassification proportion,0.2441,2,17,FALSE
1,eva_sm,Misclassification proportion,0.1417,2,17,FALSE
1,eva_fisica,Misclassification proportion,0.2014,2,17,FALSE
1,eva_transgnorma,Misclassification proportion,0.2125,2,17,FALSE
1,phys_dx_instudy,Normalized MSE (1 - R^2),0.8058,2,127,TRUE


Variable,Scale,Imputations,Mean_OOB,SD_OOB,Best_Run,Worst_Run,Runs_above_threshold,Mean_above_threshold,Historically_excluded
tipo_de_vivienda_rec2,Misclassification proportion,5,0.0797,0.0009,0.0785,0.0809,0,FALSE,FALSE
any_phys_dx,Misclassification proportion,5,0.0920,0.0001,0.0919,0.0922,0,FALSE,FALSE
evaluacindelprocesoteraputico,Misclassification proportion,5,0.0986,0.0003,0.0980,0.0989,0,FALSE,FALSE
sex_rec,Misclassification proportion,5,0.1231,0.0004,0.1224,0.1235,0,FALSE,FALSE
eva_sm,Misclassification proportion,5,0.1421,0.0003,0.1417,0.1425,0,FALSE,FALSE
eva_consumo,Misclassification proportion,5,0.1492,0.0003,0.1489,0.1496,0,FALSE,FALSE
eva_relinterp,Misclassification proportion,5,0.1640,0.0007,0.1630,0.1649,0,FALSE,FALSE
eva_fam,Misclassification proportion,5,0.1877,0.0007,0.1868,0.1887,0,FALSE,FALSE
eva_fisica,Misclassification proportion,5,0.2008,0.0004,0.2005,0.2014,0,FALSE,FALSE
eva_transgnorma,Misclassification proportion,5,0.2129,0.0005,0.2122,0.2133,0,FALSE,FALSE


Imputation,Iteration,Mean_OOB,Selected
1,1,0.3427,FALSE
1,2,0.3164,TRUE
1,3,0.3164,FALSE
2,1,0.3425,FALSE
2,2,0.3167,TRUE
2,3,0.3167,FALSE
3,1,0.3425,FALSE
3,2,0.3164,TRUE
3,3,0.3165,FALSE
4,1,0.3423,FALSE


Variable,Imputations,Mean_OOB,Runs_above_threshold,Historically_excluded,Mean_above_threshold,Review_changed_flag
tipo_de_vivienda_rec2,5,0.0797,0,FALSE,FALSE,FALSE
any_phys_dx,5,0.0920,0,FALSE,FALSE,FALSE
evaluacindelprocesoteraputico,5,0.0986,0,FALSE,FALSE,FALSE
sex_rec,5,0.1231,0,FALSE,FALSE,FALSE
eva_sm,5,0.1421,0,FALSE,FALSE,FALSE
eva_consumo,5,0.1492,0,FALSE,FALSE,FALSE
eva_relinterp,5,0.1640,0,FALSE,FALSE,FALSE
eva_fam,5,0.1877,0,FALSE,FALSE,FALSE
eva_fisica,5,0.2008,0,FALSE,FALSE,FALSE
eva_transgnorma,5,0.2129,0,FALSE,FALSE,FALSE


Imputation,Partition,Variable,Missing_before,Percent_before,Missing_after,Percent_after
1,Development,adm_age_rec3,0,0.0000,0,0
1,Development,porc_pobr,0,0.0000,0,0
1,Development,dit_m,0,0.0000,0,0
1,Development,num_trat_ant,4767,6.7597,0,0
1,Development,age_subs_onset_rec2,3441,4.8794,0,0
1,Development,sex_rec,2,0.0028,0,0
1,Development,tenure_status_household,3607,5.1148,0,0
1,Development,cohabitation,0,0.0000,0,0
1,Development,sub_dep_icd10_status,0,0.0000,0,0
1,Development,any_violence,14106,20.0026,0,0


Mean-OOB flags agree with historical exclusions. Review the number of runs above threshold at the notebook checkpoint; no variables were removed here.


Elapsed: 0.001 minutes


## Session info {#execution-session}


In [14]:
#| label: session-info
#| message: false
.t0 <- proc.time()
# Use base R session information; display loaded packages in the same browsable table style.
session <- utils::sessionInfo()
quarto_version <- system2(resolve_path(config$runtime$quarto_executable), "--version", stdout = TRUE, stderr = TRUE)
show_table(data.frame(Item = c("R", "Platform", "Operating system", "Run date", "Quarto", "Mode", "Base seed", "Threads"), Value = c(session$R.version$version.string, session$platform, session$running, format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"), paste(quarto_version, collapse = " "), config$preparation$mode, config$imputation$seed, threads)), "Session information")
session_packages <- c(session$otherPkgs, session$loadedOnly)
package_table <- do.call(rbind, lapply(session_packages, function(x) data.frame(Package = x$Package, Version = x$Version, Source = if (is.null(x$Repository)) "Installed library" else x$Repository)))
show_table(package_table[order(package_table$Package), , drop = FALSE], "Loaded R packages")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


Item,Value
R,R version 4.4.1 (2024-06-14 ucrt)
Platform,x86_64-w64-mingw32/x64
Operating system,Windows 11 x64 (build 26200)
Run date,2026-10-03 12:12:39 -03
Quarto,1.9.36
Mode,full
Base seed,2125
Threads,2


Package,Version,Source
FNN,1.1.4.1,RSPM
IRdisplay,1.1,RSPM
IRkernel,1.3.2,CRAN
Matrix,1.7-0,CRAN
R6,2.6.1,RSPM
RColorBrewer,1.1-3,RSPM
Rcpp,1.1.1,CRAN
base64enc,0.1-3,RSPM
cli,3.6.4,RSPM
codetools,0.2-20,CRAN


Elapsed: 0.006 minutes


### Files written and execution status


In [15]:
#| label: prep-written-files-and-status
#| message: false
.t0 <- proc.time()
# Report only files written during this run, with relative paths and sizes.
files <- unique(written)
manifest <- data.frame(path = substring(files, nchar(root) + 2L), bytes = as.numeric(file.info(files)$size))
show_table(manifest, "Files written to outputs during this run")
cat("Mode:", config$preparation$mode, "\n")
cat("Performance metrics and full pipeline independence: not yet established.\n")
cat(sprintf("Total elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), run_started, units = "mins"))))
process_memory <- ps::ps_memory_info()
show_table(data.frame(Measurement = names(process_memory), Bytes = as.numeric(process_memory)), "R process memory")
cat(sprintf("Elapsed: %.3f minutes\n", unname((proc.time() - .t0)["elapsed"]) / 60))


path,bytes


Mode: full 


Performance metrics and full pipeline independence: not yet established.


Total elapsed: 2.134 minutes


Measurement,Bytes
num_page_faults,5533106
peak_wset,4272541696
wset,431407104
peak_paged_pool,339104
paged_pool,329536
peak_non_paged_pool,1150992
non_paged_pool,97080
pagefile,460730368
peak_pagefile,4316033024
mem_private,460730368


Elapsed: 0.000 minutes
